In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import polars as pl
import plotly.express as px
from statsforecast import StatsForecast
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from utilsforecast.losses import *
from utilsforecast.evaluation import evaluate


from utilsforecast.losses import *

from utilsforecast.losses import *

import plotly.io as pio

from utilsforecast.losses import *
from utilsforecast.feature_engineering import fourier, pipeline
from functools import partial
from sklearn.linear_model import RidgeCV
from utilsforecast.losses import *
from plotting_utils import (
    plot_data_availability_heatmap,
    plot_missing_percentage,
    plotly_series as plot_series,
)
from statsforecast.models import SklearnModel

from statsforecast.models import (
    SeasonalNaive,
    AutoETS,
    MSTL,
)
from xgboost import XGBRegressor

# Introduction to Probabilistic Forecasting

Traditional time series forecasting methods typically provide a single predicted value for each future time point. This is known as **point forecasting**. However, real-world data is often uncertain and subject to various sources of randomness. For example, predicting tomorrow's electricity consumption or next week's sales involves many unknown factors.

**Probabilistic forecasting** addresses this uncertainty by predicting a range of possible future values, along with their associated probabilities. Instead of answering "What is the most likely value?", probabilistic forecasting answers "What is the probability that the value will fall within a certain range?".

## Why Probabilistic Forecasts Matter

- **Quantifying Uncertainty:** Probabilistic forecasts provide a measure of confidence in predictions, which is crucial for risk management and decision-making.
- **Better Decision Support:** Businesses can plan for best-case, worst-case, and most-likely scenarios.
- **Real-World Relevance:** Many applications (e.g., energy demand, finance, weather) require understanding the full range of possible outcomes, not just the average.

## Key Concepts

- **Prediction Interval:** A range within which the future value is expected to fall with a certain probability (e.g., 95% prediction interval).
- **Forecast Distribution:** The full probability distribution of possible future values, not just a single point estimate.

For example, instead of predicting that tomorrow's energy consumption will be exactly 100 kWh, a probabilistic forecast might say:

> There is a 90% chance that tomorrow's energy consumption will be between 95 and 110 kWh.

Mathematically, if $y_{t+h}$ is the value we want to forecast at time $t+h$, a probabilistic forecast provides the conditional distribution $P(y_{t+h} \mid \text{past data})$.

In the next sections, we'll explore how to generate and interpret probabilistic forecasts using modern time series tools.

In [4]:
data = pl.read_parquet(
    "data/london_smart_meters/preprocessed/london_smart_meters_merged_block_0-7.parquet"
)
timestamp = data.group_by("LCLid").agg(
    pl.datetime_range(
        start=pl.col("start_timestamp"),
        end=pl.col("start_timestamp").dt.offset_by(
            pl.format("{}m", pl.col("series_length").sub(1).mul(30))
        ),
        interval="30m",
    ).alias("ds"),
)
data = timestamp.join(data, on="LCLid", how="inner").rename(
    {"LCLid": "unique_id", "energy_consumption": "y"}
)
data.head(5)

unique_id,ds,start_timestamp,frequency,y,series_length,stdorToU,Acorn,Acorn_grouped,file,holidays,visibility,windBearing,temperature,dewPoint,pressure,apparentTemperature,windSpeed,precipType,icon,humidity,summary,__index_level_0__
str,list[datetime[ns]],datetime[ns],str,list[f64],i64,str,str,str,str,list[str],list[f64],list[i64],list[f64],list[f64],list[f64],list[f64],list[f64],list[str],list[str],list[f64],list[str],i64
"""MAC000002""","[2012-10-13 00:00:00, 2012-10-13 00:30:00, … 2014-02-27 23:30:00]",2012-10-13 00:00:00,"""30min""","[0.263, 0.269, … 1.2180001]",24144,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[13.08, 13.08, … 14.03]","[186, 186, … 200]","[8.78, 8.78, … 3.93]","[6.28, 6.28, … 1.61]","[1007.7, 1007.7, … 1004.62]","[7.55, 7.55, … 1.42]","[2.28, 2.28, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""clear-night"", ""clear-night"", … ""clear-night""]","[0.84, 0.84, … 0.85]","[""Clear"", ""Clear"", … ""Clear""]",0
"""MAC000246""","[2012-01-01 00:00:00, 2012-01-01 00:30:00, … 2014-02-27 23:30:00]",2012-01-01 00:00:00,"""30min""","[0.509, 0.317, … 0.223]",37872,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[12.99, 12.99, … 14.03]","[229, 229, … 200]","[12.12, 12.12, … 3.93]","[10.97, 10.97, … 1.61]","[1008.1, 1008.1, … 1004.62]","[12.12, 12.12, … 1.42]","[5.9, 5.9, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""partly-cloudy-night"", ""partly-cloudy-night"", … ""clear-night""]","[0.93, 0.93, … 0.85]","[""Mostly Cloudy"", ""Mostly Cloudy"", … ""Clear""]",1
"""MAC000450""","[2012-03-23 00:00:00, 2012-03-23 00:30:00, … 2014-02-27 23:30:00]",2012-03-23 00:00:00,"""30min""","[1.337, 1.426, … null]",33936,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[3.19, 3.19, … 14.03]","[78, 78, … 200]","[8.76, 8.76, … 3.93]","[7.25, 7.25, … 1.61]","[1027.41, 1027.41, … 1004.62]","[7.59, 7.59, … 1.42]","[2.18, 2.18, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""fog"", ""fog"", … ""clear-night""]","[0.9, 0.9, … 0.85]","[""Foggy"", ""Foggy"", … ""Clear""]",2
"""MAC001074""","[2012-05-09 00:00:00, 2012-05-09 00:30:00, … 2014-02-27 23:30:00]",2012-05-09 00:00:00,"""30min""","[0.18, 0.086, … null]",31680,"""ToU""","""ACORN-""","""ACORN-""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[10.51, 10.51, … 14.03]","[215, 215, … 200]","[11.46, 11.46, … 3.93]","[10.23, 10.23, … 1.61]","[1007.39, 1007.39, … 1004.62]","[11.46, 11.46, … 1.42]","[2.35, 2.35, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""partly-cloudy-night"", ""partly-cloudy-night"", … ""clear-night""]","[0.92, 0.92, … 0.85]","[""Partly Cloudy"", ""Partly Cloudy"", … ""Clear""]",3
"""MAC003223""","[2012-09-18 00:00:00, 2012-09-18 00:30:00, … 2014-02-27 23:30:00]",2012-09-18 00:00:00,"""30min""","[0.076, 0.079, … 0.38]",25344,"""Std""","""ACORN-A""","""Affluent""","""block_0""","[""NO_HOLIDAY"", ""NO_HOLIDAY"", … ""NO_HOLIDAY""]","[13.44, 13.44, … 14.03]","[236, 236, … 200]","[14.06, 14.06, … 3.93]","[10.82, 10.82, … 1.61]","[1011.09, 1011.09, … 1004.62]","[14.06, 14.06, … 1.42]","[3.86, 3.86, … 2.75]","[""rain"", ""rain"", … ""rain""]","[""clear-night"", ""clear-night"", … ""clear-night""]","[0.81, 0.81, … 0.85]","[""Clear"", ""Clear"", … ""Clear""]",4


In [5]:
id_ = "unique_id"
time_ = "ds"
target_ = "y"
id_col = pl.col(id_)
time_col = pl.col(time_)
target_col = pl.col(target_)

In [6]:
data = (
    data.filter(pl.col("file").eq("block_7"))
    .select([time_, id_, target_])
    .explode([time_, target_])
)
data.head()

ds,unique_id,y
datetime[ns],str,f64
2012-01-01 00:00:00,"""MAC000050""",0.175
2012-01-01 00:30:00,"""MAC000050""",0.212
2012-01-01 01:00:00,"""MAC000050""",0.313
2012-01-01 01:30:00,"""MAC000050""",0.302
2012-01-01 02:00:00,"""MAC000050""",0.257


In [7]:
selected_id = "MAC000193"
data = data.filter(id_col.eq(selected_id)).with_columns(
    target_col.forward_fill().backward_fill()
)
data.head()

ds,unique_id,y
datetime[ns],str,f64
2012-01-01 00:00:00,"""MAC000193""",0.368
2012-01-01 00:30:00,"""MAC000193""",0.386
2012-01-01 01:00:00,"""MAC000193""",0.17
2012-01-01 01:30:00,"""MAC000193""",0.021
2012-01-01 02:00:00,"""MAC000193""",0.038


# Bayesian Time Series Forecasting: Embracing Uncertainty with Probabilistic Models

## Introduction to Bayesian Forecasting

Bayesian time series forecasting is a powerful approach that treats all unknowns—such as model parameters and future values—as **random variables** with probability distributions. Unlike classical (frequentist) methods, which provide single "best guess" estimates, Bayesian models naturally quantify uncertainty at every step, making them ideal for **probabilistic forecasting**.

### Why Use Bayesian Methods?

- **Full Uncertainty Quantification:** Bayesian models provide a complete probability distribution for future values, not just point forecasts or symmetric intervals.
- **Flexible Modeling:** They can incorporate prior knowledge and adapt to complex, real-world data.
- **Principled Updating:** As new data arrives, Bayesian models update their beliefs in a mathematically consistent way.

---

## Key Bayesian Models for Time Series

### 1. Gaussian Process (GP) Regression

A **Gaussian Process** is a flexible, non-parametric Bayesian model for time series and regression tasks. Instead of specifying a fixed functional form, a GP defines a distribution over possible functions that fit the data.

#### How Does a Gaussian Process Work?

- **Prior:** Before seeing any data, we assume that the function values at any set of time points are jointly Gaussian distributed.
- **Covariance Function (Kernel):** The kernel encodes our assumptions about the smoothness, periodicity, or other properties of the time series.
- **Posterior:** After observing data, the GP updates its beliefs, resulting in a predictive distribution for future points.

#### Mathematical Formulation

Let $f(t)$ be the latent function describing our time series. A GP assumes:

$$
f(t) \sim \mathcal{GP}(m(t), k(t, t'))
$$

- $m(t)$: Mean function (often set to zero).
- $k(t, t')$: Covariance (kernel) function, e.g., squared exponential, periodic, etc.

Given observed data $\{(t_i, y_i)\}$, the predictive distribution for a new time $t_*$ is Gaussian:

$$
p(f(t_*) \mid \text{data}) = \mathcal{N}(\mu_*, \sigma_*^2)
$$

where $\mu_*$ and $\sigma_*^2$ are computed from the kernel and observed data.

#### Why Are GPs Powerful for Probabilistic Forecasting?

- **Uncertainty Bands:** GPs provide a mean prediction and a credible interval (uncertainty band) at every future time point.
- **Non-parametric:** They can model complex, nonlinear, and non-stationary time series without specifying a fixed equation.
- **Incorporate Prior Knowledge:** You can encode beliefs about seasonality, trends, or noise via the kernel.

#### Real-World Analogy

Imagine drawing many smooth curves that all fit your observed data. The GP tells you, for each future time, how much those curves tend to agree (narrow uncertainty) or disagree (wide uncertainty), giving you a full distribution of possible futures.

---

### 2. Gaussian Markov Models (State Space Models)

A **Gaussian Markov Model** is a type of **state space model** where the hidden states and observations are assumed to be Gaussian and the system evolves according to Markovian (memoryless) dynamics.

#### The Classic Example: Kalman Filter

The **Kalman Filter** is a foundational Gaussian Markov model for time series. It models the system as:

- **State Equation:** $x_{t+1} = A x_t + w_t$, where $w_t \sim \mathcal{N}(0, Q)$
- **Observation Equation:** $y_t = H x_t + v_t$, where $v_t \sim \mathcal{N}(0, R)$

Here, $x_t$ is the hidden state, $y_t$ is the observed value, and $A$, $H$, $Q$, $R$ are model parameters.

#### Probabilistic Forecasting with Kalman Filters

- At each time step, the Kalman filter provides a **predictive distribution** for the next observation, not just a point estimate.
- The forecast is Gaussian, with a mean and variance that reflect both process and measurement uncertainty.

#### Why Are Gaussian Markov Models Useful?

- **Recursive Updates:** Efficiently update forecasts as new data arrives.
- **Handles Missing Data:** Naturally accommodates gaps in observations.
- **Extensible:** Can be extended to nonlinear or non-Gaussian cases (e.g., particle filters, extended Kalman filters).

---

## Bayesian vs. Classical Probabilistic Forecasting

| Feature                | Bayesian Models (GP, Kalman) | Classical Models (ETS, ARIMA) |
|------------------------|------------------------------|-------------------------------|
| Uncertainty Quantified | Full posterior distribution  | Analytical/simulation-based   |
| Prior Knowledge        | Easily incorporated          | Harder to include             |
| Flexibility            | Very high                    | Moderate                      |
| Output                 | Predictive distribution      | Point + interval              |
| Computational Cost     | Often higher                 | Usually lower                 |

---

## Practical Considerations

- **Computation:** Bayesian models, especially GPs, can be computationally intensive for large datasets. Approximations and sparse methods are often used.
- **Interpretability:** Bayesian credible intervals have a direct probabilistic interpretation: "There is a 95% probability that the future value lies within this interval, given the data and model."
- **Software:** Libraries like `gpytorch`, `scikit-learn` (for GPs), and `pykalman` (for Kalman filters) make Bayesian time series modeling accessible in Python.

---

## Summary

- **Bayesian time series models** like Gaussian Processes and Gaussian Markov (state space) models provide a principled, flexible framework for probabilistic forecasting.
- They deliver not just point forecasts, but full predictive distributions, allowing you to quantify and communicate uncertainty in a rigorous way.
- Bayesian methods are especially valuable when you want to incorporate prior knowledge, handle complex data, or need robust uncertainty estimates for decision-making.

---

**In summary:**  
Bayesian approaches, such as Gaussian Processes and Gaussian Markov models, are at the forefront of probabilistic time series forecasting. They empower you to move beyond simple intervals and embrace the full spectrum of uncertainty—making your forecasts more informative, honest, and actionable.

In [ ]:
from functools import partial
from metrics_utils import winkler_score

metrics = [
    mqloss,
    winkler_score,
    coverage,
]
evaluate(
    y_hat,
    metrics=metrics,
    level=[80],
)

unique_id,metric,XGBRegressor
str,str,f64
"""MAC000193""","""mqloss""",0.059518
"""MAC000193""","""winkler_score_level80""",1.190367
"""MAC000193""","""coverage_level80""",0.708333
